In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_silver.regional_production AS
SELECT
  duoarea,
  MAX(area_name) AS area_name,
  CAST(CONCAT(period, '-01') AS DATE) AS month_start,
  MAX(value) AS production_mbbl_d
FROM workspace.petropulse_bronze.eia_production_raw
WHERE process = 'FPF'
  AND units = 'MBBL/D'
  AND (duoarea <> 'SAK' OR series = 'MANFPAK2')
GROUP BY duoarea, CAST(CONCAT(period, '-01') AS DATE);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_silver.seismic_events AS
SELECT event_id, magnitude, place, event_time, latitude, longitude, depth_km
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY updated_ms DESC, ingested_at DESC) AS rn
  FROM workspace.petropulse_bronze.usgs_seismic_raw
  WHERE event_type = 'earthquake' AND magnitude IS NOT NULL
)
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_silver.weather_latest AS
SELECT region_id, observed_at, temperature_c, wind_speed_kmh, wind_gust_kmh, precipitation_mm
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY region_id ORDER BY observed_at DESC, ingested_at DESC) AS rn
  FROM workspace.petropulse_bronze.openmeteo_weather_raw
)
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_silver.region_seismic_nearby AS
WITH cfg AS (
  SELECT
    MAX(CASE WHEN param_name = 'seismic_radius_km'     THEN param_value END) AS radius_km,
    MAX(CASE WHEN param_name = 'seismic_lookback_days' THEN param_value END) AS lookback_days
  FROM workspace.petropulse_ref.risk_thresholds
),
pairs AS (
  SELECT
    r.region_id, e.event_id, e.magnitude, e.event_time, e.place,
    2 * 6371 * ASIN(SQRT(
      POW(SIN(RADIANS(e.latitude - r.latitude) / 2), 2) +
      COS(RADIANS(r.latitude)) * COS(RADIANS(e.latitude)) *
      POW(SIN(RADIANS(e.longitude - r.longitude) / 2), 2)
    )) AS distance_km
  FROM workspace.petropulse_ref.operating_regions r
  CROSS JOIN workspace.petropulse_silver.seismic_events e
  CROSS JOIN cfg
  WHERE e.event_time >= current_timestamp() - make_interval(0, 0, 0, CAST(cfg.lookback_days AS INT))
)
SELECT p.*
FROM pairs p CROSS JOIN cfg
WHERE p.distance_km <= cfg.radius_km;

In [0]:
%sql
SELECT 'seismic_events' AS tbl, COUNT(*) AS n FROM workspace.petropulse_silver.seismic_events
UNION ALL SELECT 'weather_latest', COUNT(*) FROM workspace.petropulse_silver.weather_latest
UNION ALL SELECT 'region_seismic_nearby', COUNT(*) FROM workspace.petropulse_silver.region_seismic_nearby;

In [0]:
%sql
SELECT region_id, COUNT(*) AS events, ROUND(MAX(magnitude),1) AS max_mag, ROUND(MIN(distance_km)) AS closest_km
FROM workspace.petropulse_silver.region_seismic_nearby
GROUP BY region_id
ORDER BY region_id;

In [0]:
%sql
SELECT duoarea, month_start, production_mbbl_d
FROM workspace.petropulse_silver.regional_production
WHERE duoarea = 'SAK' AND month_start = '2026-06-01';